In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix
import os

# 1. Load our newly compressed data and labels
data_dir = "/workspace/data/processed/geo_pool"
print("[*] Loading PCA Matrix and Labels...")
X = pd.read_csv(os.path.join(data_dir, "X_matrix_pca_reduced.csv"), index_col=0)
y = pd.read_csv(os.path.join(data_dir, "y_labels.csv"), index_col=0)

# Ensure y is a 1D array (pandas Series)
y = y.iloc[:, 0]

# 2. The Train/Test Split (80% Training, 20% Testing)
# 'stratify=y' ensures we keep the same ratio of survivors to non-survivors in both splits!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"[+] Training Set: {X_train.shape[0]} patients")
print(f"[+] Testing Set (The Vault): {X_test.shape[0]} patients")

print("\n[*] Calculating Class Imbalance Weight...")
# Count how many survivors (0) and non-survivors (1) are in our training data
survivors_count = (y_train == 0).sum()
deaths_count = (y_train == 1).sum()
imbalance_ratio = survivors_count / deaths_count

print(f"[+] There are {survivors_count} survivors for every {deaths_count} deaths.")
print(f"[+] Setting scale_pos_weight to: {imbalance_ratio:.2f}")

print("\n[*] Training the XGBoost 'Geneticist' Model (Weighted)...")
model = XGBClassifier(
    n_estimators=500,        
    learning_rate=0.05,      
    max_depth=4,             
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=imbalance_ratio  # <--- THIS IS THE MAGIC FIX
)

model.fit(X_train, y_train)
print("[+] Training Complete!")

# 4. Administer the Final Exam (Testing the model on unseen data)
print("\n[*] Grading the Model on the 20% Test Set...")
y_pred = model.predict(X_test)                  # Hard predictions (0 or 1)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probabilities (e.g., 85% chance of mortality)

# 5. Calculate our Medical Benchmark Metrics
auc_score = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\n==============================")
print(f"      FINAL EXAM RESULTS      ")
print(f"==============================")
print(f"ROC-AUC Score: {auc_score:.4f} (1.0 is perfect, 0.5 is guessing)")
print(f"F1 Score:      {f1:.4f} (Balances false positives and false negatives)")
print(f"==============================\n")

print("Detailed Medical Report:")
print(classification_report(y_test, y_pred))

[*] Loading PCA Matrix and Labels...
[+] Training Set: 624 patients
[+] Testing Set (The Vault): 156 patients

[*] Calculating Class Imbalance Weight...
[+] There are 533 survivors for every 91 deaths.
[+] Setting scale_pos_weight to: 5.86

[*] Training the XGBoost 'Geneticist' Model (Weighted)...
[+] Training Complete!

[*] Grading the Model on the 20% Test Set...

      FINAL EXAM RESULTS      
ROC-AUC Score: 0.6447 (1.0 is perfect, 0.5 is guessing)
F1 Score:      0.7847 (Balances false positives and false negatives)

Detailed Medical Report:
              precision    recall  f1-score   support

           0       0.85      1.00      0.92       133
           1       0.00      0.00      0.00        23

    accuracy                           0.85       156
   macro avg       0.43      0.50      0.46       156
weighted avg       0.73      0.85      0.78       156



/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [3]:
import numpy as np
from sklearn.metrics import precision_recall_curve

print("[*] Extracting raw probabilities instead of hard guesses...")
# y_pred_proba already contains the raw % chance of mortality from your previous cell
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Calculate the F1 score for every single possible threshold between 0.0 and 1.0
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

# Find the exact threshold that gives us the highest F1 score
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\n[+] XGBoost's Default Threshold: 0.5000")
print(f"[+] The ACTUAL Optimal Threshold: {optimal_threshold:.4f}")

print(f"\n[*] Recalculating predictions using the {optimal_threshold:.4f} cutoff...")
# Force the model to use OUR threshold, not its default
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

print("\n==============================")
print(f"  ADJUSTED MEDICAL REPORT     ")
print("==============================")
print(classification_report(y_test, y_pred_optimal))

[*] Extracting raw probabilities instead of hard guesses...

[+] XGBoost's Default Threshold: 0.5000
[+] The ACTUAL Optimal Threshold: 0.0562

[*] Recalculating predictions using the 0.0562 cutoff...

  ADJUSTED MEDICAL REPORT     
              precision    recall  f1-score   support

           0       0.89      0.87      0.88       133
           1       0.35      0.39      0.37        23

    accuracy                           0.80       156
   macro avg       0.62      0.63      0.62       156
weighted avg       0.81      0.80      0.81       156

